# 🏙️ Bengaluru House Price Prediction – End‑to‑End Tutorial

This notebook walks through the complete workflow for the hackathon:

1. **Load & understand the data** (train, test, and external datasets)
2. **Clean & preprocess** the data (handle missing values, engineer features)
3. **Build & compare models** (Linear Regression, Ridge, Lasso, Random Forest)
4. **Train the best model** on full data and **create the submission CSV** (`ID`, `Price`).

We’ll keep the code readable and well‑commented so it feels like a step‑by‑step mini‑course. 💻✨

## 🔹 Step 1: Load all datasets

We have four CSV files:
- `train.csv` – training data with target column `price`
- `test.csv` – same structure but **without** `price`
- `avg_rent.csv` – average 2BHK rent for each location
- `dist_from_city_centre.csv` – distance of each location from city centre

We'll load them using **pandas** and take a quick look.

In [3]:
import pandas as pd
import numpy as np

train = pd.read_csv("C:\\Users\\B Ramesh\\OneDrive\\Desktop\\ML Hackathon\\train_(2)_(1)_(1).csv")
test = pd.read_csv("C:\\Users\\B Ramesh\\OneDrive\\Desktop\\ML Hackathon\\test_(2)_(1)_(1).csv")
avg_rent = pd.read_csv("C:\\Users\\B Ramesh\\OneDrive\\Desktop\\ML Hackathon\\avg_rent_(1)_(1)_(1).csv")
dist = pd.read_csv("C:\\Users\\B Ramesh\\OneDrive\\Desktop\\ML Hackathon\\dist_from_city_centre_(1).csv")

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('\nTrain head:')
display(train.head())
print('\nTest head:')
display(test.head())
print('\nAverage Rent head:')
display(avg_rent.head())
print('\nDistance from City Centre head:')
display(dist.head())

Train shape: (10656, 10)
Test shape : (2664, 9)

Train head:


,ID,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00



Test head:


,ID,area_type,availability,location,size,society,total_sqft,bath,balcony
0,0,Super built-up Area,Ready To Move,Chamrajpet,2 BHK,NaN,650,1.0,1.0
1,1,Super built-up Area,Ready To Move,7th Phase JP Nagar,3 BHK,SrncyRe,1370,2.0,1.0
2,2,Super built-up Area,Ready To Move,Whitefield,3 BHK,AjhalNa,1725,3.0,2.0
3,3,Built-up Area,Ready To Move,Jalahalli,2 BHK,NaN,1000,2.0,0.0
4,4,Plot Area,Ready To Move,TC Palaya,1 Bedroom,NaN,1350,1.0,0.0



Average Rent head:


,location,avg_2bhk_rent
0,Krishnarajapura,11954
1,Sarjapur,45000
2,Whitefield Hope Farm Junction,26370
3,Devanahalli,17302
4,Whitefield,14981



Distance from City Centre head:


,location,dist_from_city
0,Whitefield,17.3
1,Sarjapur Road,17.2
2,Electronic City,18.1
3,Kanakpura Road,26.5
4,Thanisandra,11.5


## 🔹 Step 2: Explore data & check missing values

Before cleaning, we need to understand:
- Column data types
- How many missing values each column has
- If any column looks too noisy or useless

We'll start with `.info()` and `.isnull().sum()`.

In [5]:
print('Train info:')
display(train.info())

print('\nTrain missing values:')
display(train.isnull().sum())

print('\nTest info:')
display(test.info())

print('\nTest missing values:')
display(test.isnull().sum())

Train info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10656 entries, 0 to 10655
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            10656 non-null  int64  
 1   area_type     10656 non-null  object 
 2   availability  10656 non-null  object 
 3   location      10655 non-null  object 
 4   size          10642 non-null  object 
 5   society       6228 non-null   object 
 6   total_sqft    10656 non-null  object 
 7   bath          10591 non-null  float64
 8   balcony       10152 non-null  float64
 9   price         10656 non-null  float64
dtypes: float64(3), int64(1), object(6)
memory usage: 832.6+ KB


None


Train missing values:


ID                 0
area_type          0
availability       0
location           1
size              14
society         4428
total_sqft         0
bath              65
balcony          504
price              0
dtype: int64


Test info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2664 entries, 0 to 2663
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            2664 non-null   int64  
 1   area_type     2664 non-null   object 
 2   availability  2664 non-null   object 
 3   location      2664 non-null   object 
 4   size          2662 non-null   object 
 5   society       1590 non-null   object 
 6   total_sqft    2664 non-null   object 
 7   bath          2656 non-null   float64
 8   balcony       2559 non-null   float64
dtypes: float64(2), int64(1), object(6)
memory usage: 187.4+ KB


None


Test missing values:


ID                 0
area_type          0
availability       0
location           0
size               2
society         1074
total_sqft         0
bath               8
balcony          105
dtype: int64

### 2.1 Inspect tricky columns

Based on the description + info, a few columns need special handling:

- **`size`** – values like `'2 BHK'`, `'3 Bedroom'` → we’ll extract the number to create a numeric `bhk` feature.
- **`total_sqft`** – stored as text, may contain ranges (`'1200-1500'`) or units. We'll convert this to a clean numeric feature.
- **`availability`** – contains labels like `'Ready To Move'`, `'Immediate'`, and date‑like strings such as `'19-Dec'`. We'll make:
  - a binary feature `ready_to_move`
  - a numeric `possession_year` for date‑like entries.
- **`society`** – has a lot of missing values; it’s mostly an encrypted ID, not very informative → we’ll drop it.

Let’s quickly peek at the patterns in `size`, `total_sqft`, and `availability`.

In [7]:
print('Unique size values (top 10):')
display(train['size'].value_counts().head(10))

print('\nRandom total_sqft samples:')
display(train['total_sqft'].sample(15, random_state=42))

print('\nAvailability value counts (top 10):')
display(train['availability'].value_counts().head(10))

Unique size values (top 10):


size
2 BHK        4146
3 BHK        3425
4 Bedroom     659
4 BHK         478
3 Bedroom     446
1 BHK         435
2 Bedroom     281
5 Bedroom     232
6 Bedroom     154
1 Bedroom      88
Name: count, dtype: int64


Random total_sqft samples:


1350    1079
2012    1765
1965    1358
9800    3335
251     1060
33      1693
5563    1475
4121    3100
4793    1680
265     1600
8029     873
2865    1015
914      918
7172    2365
1479    1300
Name: total_sqft, dtype: object


Availability value counts (top 10):


availability
Ready To Move    8468
18-Dec            237
18-May            233
18-Apr            204
18-Aug            162
19-Dec            152
18-Jul            115
18-Mar            108
18-Jun             79
20-Dec             77
Name: count, dtype: int64

## 🔹 Step 2.2: Define cleaning & feature engineering functions

We'll now write reusable functions so the **same logic** applies to both `train` and `test`.

What we’ll do:
1. **Clean locations** → `location_clean` (lowercase, stripped spaces)
2. **Extract `bhk`** from the `size` text
3. **Convert `total_sqft`** to numeric:
   - If it's a range like `'1200-1500'` → take the average
   - If it starts with a number and then text → take the numeric part
   - If it’s messy/unreadable → set to NaN
4. **Engineer availability features**:
   - `ready_to_move` = 1 if `'Ready To Move'` or `'Immediate'`, else 0
   - `possession_year` from strings like `'18-Aug'` (year of possession)
5. **Handle missing `bath` and `balcony`** using median imputation
6. **Drop weak / high‑missing columns** (like `society`)
7. **Merge external features** (`avg_2bhk_rent`, `dist_from_city`) using cleaned location.
8. **Create some ratio features** like `bath_per_bhk` and `balcony_per_bhk`.

Let's implement this step by step.

In [9]:
def clean_location(s):
    """Lowercase and strip extra spaces from location strings."""
    if isinstance(s, str):
        return s.strip().lower()
    return s

def extract_bhk(size_value):
    """Extract numeric BHK/Bedroom count from size column (e.g. '2 BHK', '3 Bedroom')."""
    if isinstance(size_value, str):
        parts = size_value.split()
        for p in parts:
            if p.isdigit():
                return float(p)
    return np.nan

def convert_total_sqft(x):
    """Convert total_sqft text to a numeric value.
    - If it's a range '1200-1500' → return average
    - Else → return the leading numeric part
    - If can't parse → return NaN
    """
    if isinstance(x, str):
        x = x.strip()
        if '-' in x:  # handle ranges
            parts = x.split('-')
            try:
                nums = [float(p) for p in parts]
                return sum(nums) / len(nums)
            except ValueError:
                return np.nan
        # extract leading numeric token
        num_str = ''
        for ch in x:
            if ch.isdigit() or ch == '.':
                num_str += ch
            elif num_str:
                break
        try:
            return float(num_str)
        except ValueError:
            return np.nan
    return x

def parse_possession_year(val):
    """Parse year from availability like '18-Aug'. Non-date labels → NaN."""
    if not isinstance(val, str):
        return np.nan
    if val in ['Ready To Move', 'Immediate']:
        return np.nan
    try:
        dt = pd.to_datetime(val, format='%y-%b')
        return dt.year
    except Exception:
        return np.nan

def base_preprocess(df):
    """Core cleaning steps shared between train and test (without merging)."""
    df = df.copy()

    # 1) Clean location
    df['location'] = df['location'].fillna('missing_location')
    df['location_clean'] = df['location'].apply(clean_location)

    # 2) BHK from size
    df['bhk'] = df['size'].apply(extract_bhk)
    # size has few missing values → fill bhk with median
    df['bhk'] = df['bhk'].fillna(df['bhk'].median())

    # 3) Clean total_sqft
    df['total_sqft_num'] = df['total_sqft'].apply(convert_total_sqft)

    # 4) Availability features
    df['ready_to_move'] = df['availability'].isin(['Ready To Move', 'Immediate']).astype(int)
    df['possession_year'] = df['availability'].apply(parse_possession_year)

    # 5) Drop high-missing / low-signal columns
    if 'society' in df.columns:
        df = df.drop(columns=['society'])

    # 6) Handle bath & balcony using median
    if 'bath' in df.columns:
        df['bath'] = df['bath'].fillna(df['bath'].median())
    if 'balcony' in df.columns:
        df['balcony'] = df['balcony'].fillna(df['balcony'].median())

    # 7) Ratio features
    df['bath_per_bhk'] = df['bath'] / df['bhk']
    df['balcony_per_bhk'] = df['balcony'] / df['bhk']

    return df

### 2.3 Apply cleaning & merge external datasets

Now we’ll:

1. Apply `base_preprocess` to both `train` and `test`.
2. Prepare external datasets with cleaned location.
3. Merge `avg_2bhk_rent` and `dist_from_city` into both train and test.
4. Check missing values **after** merging.

In [11]:
# Apply base preprocessing
train_pp = base_preprocess(train)
test_pp = base_preprocess(test)

# Clean external datasets' locations
avg_rent_pp = avg_rent.copy()
avg_rent_pp['location_clean'] = avg_rent_pp['location'].apply(clean_location)
avg_rent_pp = avg_rent_pp.drop(columns=['location'])

dist_pp = dist.copy()
dist_pp['location_clean'] = dist_pp['location'].apply(clean_location)
dist_pp = dist_pp.drop(columns=['location'])

# Merge external datasets into train and test
train_pp = train_pp.merge(avg_rent_pp, on='location_clean', how='left')
train_pp = train_pp.merge(dist_pp, on='location_clean', how='left')

test_pp = test_pp.merge(avg_rent_pp, on='location_clean', how='left')
test_pp = test_pp.merge(dist_pp, on='location_clean', how='left')

print('Train (after merge) shape:', train_pp.shape)
print('Test (after merge) shape :', test_pp.shape)

print('\nMissing values in merged train (top 20):')
display(train_pp.isnull().sum().sort_values(ascending=False).head(20))

Train (after merge) shape: (10656, 18)
Test (after merge) shape : (2664, 17)

Missing values in merged train (top 20):


possession_year    8482
avg_2bhk_rent      6932
dist_from_city     1006
size                 14
bhk                   0
balcony_per_bhk       0
bath_per_bhk          0
ready_to_move         0
total_sqft_num        0
ID                    0
area_type             0
price                 0
balcony               0
bath                  0
total_sqft            0
location              0
availability          0
location_clean        0
dtype: int64

### 2.4 Handle remaining missing values & reduce location cardinality

After merging external datasets, some locations may still have missing:
- `avg_2bhk_rent` (if that location isn’t present in `avg_rent.csv`)
- `dist_from_city` (if not present in distance file)

We’ll handle them as follows:
- Fill missing `avg_2bhk_rent` with **median** rent
- Fill missing `dist_from_city` with **median** distance

Also, the number of unique locations is huge (1000+), which can make one‑hot encoding very sparse.

So we’ll:
- Count how many rows each `location_clean` has in the **train** data
- Mark locations with very low counts (e.g., `<= 20`) as `'other'`
- Use a new feature `location_grouped` with this reduced set

This helps the model generalize better and reduces noise from rare locations.

In [13]:
# Fill missing external numeric features with median
for col in ['avg_2bhk_rent', 'dist_from_city']:
    if col in train_pp.columns:
        median_val = train_pp[col].median()
        train_pp[col] = train_pp[col].fillna(median_val)
        test_pp[col] = test_pp[col].fillna(median_val)

# Group locations
location_counts = train_pp['location_clean'].value_counts()
rare_locs = location_counts[location_counts <= 20].index

def group_location(loc):
    if loc in rare_locs:
        return 'other'
    return loc

train_pp['location_grouped'] = train_pp['location_clean'].apply(group_location)
test_pp['location_grouped'] = test_pp['location_clean'].apply(group_location)

print('Unique locations (original):', train_pp['location_clean'].nunique())
print('Unique locations (grouped):', train_pp['location_grouped'].nunique())

Unique locations (original): 1180
Unique locations (grouped): 120


### 2.5 Optional: Handle obvious outliers

We’ll create a **price per sqft** feature and remove extreme outliers that can hurt a regression model:

- Compute `price_per_sqft = price * 100000 / total_sqft_num` (since price is in Lakhs)
- Drop rows where `total_sqft_num` is very small or missing
- Remove extreme `price_per_sqft` values outside the 1st–99th percentile range

This keeps the model from chasing crazy values and usually improves RMSE.

In [15]:
# Remove rows with invalid / missing total_sqft
train_pp = train_pp[train_pp['total_sqft_num'].notnull()]
train_pp = train_pp[train_pp['total_sqft_num'] > 300]  # tiny houses are suspicious

# Create price_per_sqft just for outlier detection (do NOT use as feature)
train_pp['price_per_sqft'] = (train_pp['price'] * 100000) / train_pp['total_sqft_num']

q1 = train_pp['price_per_sqft'].quantile(0.01)
q99 = train_pp['price_per_sqft'].quantile(0.99)
train_pp = train_pp[(train_pp['price_per_sqft'] >= q1) & (train_pp['price_per_sqft'] <= q99)]

# Drop the helper column so it is not used as a feature (prevents leakage)
train_pp = train_pp.drop(columns=['price_per_sqft'])

print('Train shape after outlier removal:', train_pp.shape)


Train shape after outlier removal: (10432, 19)


## 🔹 Step 3: Model building – Linear, Ridge, Lasso, Random Forest

Now that our dataset is cleaned and enriched, we’ll:

1. Define our **feature matrix** `X` and **target** `y` (`price`).
2. Split the training data into **train** and **validation** sets.
3. Build a preprocessing + model pipeline using **scikit‑learn**:
   - Numeric features → impute missing values (median) + scale
   - Categorical features → impute (most frequent) + one‑hot encode
4. Train and evaluate these models (using RMSE):
   - Linear Regression
   - Ridge Regression
   - Lasso Regression
   - Random Forest Regressor

Then we’ll pick the model with the **lowest validation RMSE**.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Define target and features ---
target = 'price'

# We do NOT want to use these columns as predictors
# (ID is just an identifier, price is target, raw text location columns are replaced by location_grouped)
drop_cols = ['price', 'location', 'location_clean']

# price_per_sqft was already dropped in previous step, so it will not appear here
feature_cols = [c for c in train_pp.columns if c not in drop_cols]

print("Number of feature columns:", len(feature_cols))

X = train_pp[feature_cols]
y = train_pp[target]

# --- Identify numeric and categorical columns ---
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
# ID is not really a predictive numeric feature, so exclude it if present
if 'ID' in numeric_features:
    numeric_features.remove('ID')

categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# --- Preprocessing pipelines ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


Number of feature columns: 16
Numeric features: ['bath', 'balcony', 'bhk', 'total_sqft_num', 'possession_year', 'bath_per_bhk', 'balcony_per_bhk', 'avg_2bhk_rent', 'dist_from_city']
Categorical features: ['area_type', 'availability', 'size', 'total_sqft', 'location_grouped']


In [18]:
# --- Train / validation split ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Define models ---
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),                    # keep it simple for older sklearn
    'Lasso': Lasso(alpha=0.001, max_iter=10000),  # compatible with older versions
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
}

results = {}

# --- Train and evaluate each model ---
for name, model in models.items():
    print(f"\nTraining model: {name}")
    pipe = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)

    # Older sklearn versions do not support squared=False, so compute RMSE manually
    mse = mean_squared_error(y_val, preds)
    rmse = np.sqrt(mse)

    results[name] = rmse
    print(f'Validation RMSE for {name}: {rmse:.4f}')

print('\nModel comparison (lower RMSE is better):')
for name, rmse in results.items():
    print(f'{name}: {rmse:.4f}')



Training model: LinearRegression
Validation RMSE for LinearRegression: 64.1544

Training model: Ridge
Validation RMSE for Ridge: 60.5121

Training model: Lasso
Validation RMSE for Lasso: 65.4949

Training model: RandomForest
Validation RMSE for RandomForest: 52.3065

Model comparison (lower RMSE is better):
LinearRegression: 64.1544
Ridge: 60.5121
Lasso: 65.4949
RandomForest: 52.3065


In [19]:
# Quick check: list the final feature columns being used
print("Number of feature columns used for modelling:", len(feature_cols))
print("Sample feature columns:", feature_cols[:20])


Number of feature columns used for modelling: 16
Sample feature columns: ['ID', 'area_type', 'availability', 'size', 'total_sqft', 'bath', 'balcony', 'bhk', 'total_sqft_num', 'ready_to_move', 'possession_year', 'bath_per_bhk', 'balcony_per_bhk', 'avg_2bhk_rent', 'dist_from_city', 'location_grouped']


We now have RMSE scores for all four models. Let’s pick the **best one** (smallest RMSE) and retrain it on the **full training data**.

## 🔹 Step 4: Train best model on full data & create submission

Steps:

1. Pick the model with the **lowest validation RMSE**.
2. Re‑create a pipeline with `preprocess` + `best_model`.
3. Fit on **all cleaned training data** (`train_pp`).
4. Apply the same preprocessing to `test_pp` and generate predictions.
5. Build a submission DataFrame with columns:
   - `ID` → from `test` dataset
   - `Price` → model prediction
6. Save to `submission.csv`.

In [22]:
# FINAL MODEL TRAINING ON FULL CLEANED DATA (RandomForest)
# --- Pick the best model based on RMSE ---
best_model_name = min(results, key=results.get)
print("Best model based on validation RMSE:", best_model_name)

best_model = models[best_model_name]   # RandomForest in your case

# --- Prepare full training data ---
X_full = train_pp[feature_cols]
y_full = train_pp['price']

# --- Build full pipeline ---
final_pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', best_model)
])

# --- Train model on FULL cleaned dataset ---
print("\nTraining final model on FULL dataset...")
final_pipeline.fit(X_full, y_full)
print("Done! 🌟")


Best model based on validation RMSE: RandomForest

Training final model on FULL dataset...
Done! 🌟


In [34]:
#Generate Predictions & Create Submission File.
# Prepare test data using the same feature columns
X_test_final = test_pp[feature_cols]

# Predict
final_preds = final_pipeline.predict(X_test_final)

# Create submission file
submission = pd.DataFrame({
    'ID': test_pp['ID'],
    'Price': final_preds
})

submission_path = r"C:\Users\B Ramesh\Desktop\submission.csv"
submission.to_csv("my_submission_final.csv", index=False, encoding='utf-8')


print("Submission file created at:", submission_path)

submission.head()


Submission file created at: C:\Users\B Ramesh\Desktop\submission.csv


,ID,Price
0,0,53.143244
1,1,62.339328
2,2,109.987582
3,3,44.202911
4,4,70.598378


### ✅ Done!

- The notebook:
  - Loads and cleans all datasets
  - Engineers useful features
  - Trains 4 different regression models
  - Selects the best one using validation RMSE
  - Generates `submission.csv` with columns `ID` and `Price`

You can now upload `submission.csv` to the hackathon platform along with this notebook (`.ipynb`) as your **submission code**. 🚀